<a href="https://colab.research.google.com/github/SandhyaSharma1512/2025ae05678-mlops-assignment1/blob/main/notebooks/02_model_training_mlflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import os

# Check if the repository is already cloned
repo_name = "2025ae05678-mlops-assignment1"
if not os.path.exists(os.path.join(os.getcwd(), "src")):
    # Clone the repository if not already cloned
    !git clone https://github.com/SandhyaSharma1512/2025ae05678-mlops-assignment1 .
else:
    print(f"Repository '{repo_name}' already exists. Skipping clone step.")

# Verify that the 'src' folder is visible in the current workspace
print("Current working directory:", os.getcwd())
print("Directory contents:", os.listdir('.'))

fatal: destination path '.' already exists and is not an empty directory.
Current working directory: /home/s2025ae05678/workspace/2025ae05678-mlops-assignment1/notebooks
Directory contents: ['2025ae05678-mlops-assignment1', '03_inference_api.ipynb', '01_eda.ipynb', '02_model_training_mlflow.ipynb']


# 02 — Feature Engineering, Model Training, Evaluation, and MLflow

preprocessing, two classification models, stratified cross-validation, evaluation metrics, experiment tracking, and saving a reusable preprocessing-plus-model pipeline.

Run from the project root. Results depend on the dataset version and installed package versions; use the metrics produced by your actual run in the report.


In [14]:
pip install -r ../requirements.txt

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 294 kB 8.4 MB/s            
     |████████████████████████████████| 64 kB 3.9 MB/s             
     |████████████████████████████████| 365 kB 54.9 MB/s            
     |████████████████████████████████| 73 kB 3.3 MB/s             
     |████████████████████████████████| 10.4 MB 30.9 MB/s            
     |████████████████████████████████| 457 kB 54.7 MB/s            
     |████████████████████████████████| 487 kB 49.5 MB/s            
     |████████████████████████████████| 181 kB 22.8 MB/s            
     |████████████████████████████████| 3.8 MB 52.8 MB/s            
     |████████████████████████████████| 78 kB 10.6 MB/s            
  Attempting uninstall: h11
    Found existing installation: h11 0.14.0
    Uninstalling h11-0.14.0:
      Successfully uninstalled h11-0.14.0
Note: you may need to restart the kernel to use updated packages.


In [15]:
from pathlib import Path
import sys, json

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from heart_disease.data import download_dataset, prepare_dataset, RAW_PATH, PROCESSED_PATH
if not RAW_PATH.exists():
    download_dataset(RAW_PATH)
prepare_dataset(RAW_PATH, PROCESSED_PATH)
print("Dataset ready:", PROCESSED_PATH)


Dataset ready: /home/s2025ae05678/workspace/2025ae05678-mlops-assignment1/data/processed/heart_disease.csv


/home/s2025ae05678/workspace/2025ae05678-mlops-assignment1/src/heart_disease/data.py:33: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  frame[col] = pd.to_numeric(frame[col], errors="ignore")


In [16]:
import pandas as pd
df = pd.read_csv(PROCESSED_PATH)
X = df.drop(columns=["target"])
y = df["target"].astype(int)
print("Features:", X.shape, "Target counts:", y.value_counts().to_dict())


Features: (303, 13) Target counts: {0: 164, 1: 139}


## Preprocessing strategy

The reusable pipeline imputes missing numeric values with the median, scales numeric features, imputes categorical values with the most frequent value, and one-hot encodes categorical features. The preprocessing is fit only on training folds through the scikit-learn `Pipeline`, helping prevent data leakage.


In [17]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from heart_disease.train import build_pipeline, RANDOM_STATE

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {"accuracy": "accuracy", "precision": "precision", "recall": "recall", "roc_auc": "roc_auc"}


In [18]:
candidates = {
    "logistic_regression": LogisticRegression(
        max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE
    ),
    "random_forest": RandomForestClassifier(
        n_estimators=300, min_samples_leaf=2, class_weight="balanced",
        random_state=RANDOM_STATE
    ),
}

rows = []
fitted_models = {}
for name, estimator in candidates.items():
    pipeline = build_pipeline(X_train, estimator)
    cv_results = cross_validate(pipeline, X_train, y_train, cv=cv, scoring=scoring, error_score="raise")
    pipeline.fit(X_train, y_train)
    pred = pipeline.predict(X_test)
    prob = pipeline.predict_proba(X_test)[:, 1]
    row = {
        "model": name,
        "cv_accuracy_mean": cv_results["test_accuracy"].mean(),
        "cv_precision_mean": cv_results["test_precision"].mean(),
        "cv_recall_mean": cv_results["test_recall"].mean(),
        "cv_roc_auc_mean": cv_results["test_roc_auc"].mean(),
        "test_accuracy": accuracy_score(y_test, pred),
        "test_precision": precision_score(y_test, pred, zero_division=0),
        "test_recall": recall_score(y_test, pred, zero_division=0),
        "test_roc_auc": roc_auc_score(y_test, prob),
    }
    rows.append(row)
    fitted_models[name] = pipeline

results_df = pd.DataFrame(rows).sort_values("cv_roc_auc_mean", ascending=False)
display(results_df)


,model,cv_accuracy_mean,cv_precision_mean,cv_recall_mean,cv_roc_auc_mean,test_accuracy,test_precision,test_recall,test_roc_auc
0,logistic_regression,0.813946,0.815373,0.773913,0.896305,0.868852,0.812500,0.928571,0.950216
1,random_forest,0.793027,0.789518,0.755731,0.882836,0.918033,0.870968,0.964286,0.954545


## Model selection

The example selects the model with the highest mean cross-validation ROC-AUC. This is a baseline choice, not a universal rule. For screening-style problems, discuss false negatives and recall, calibration, decision thresholds, class imbalance, and the need for clinical/domain expertise. Do not claim clinical suitability from these metrics.


In [19]:
best_name = results_df.iloc[0]["model"]
best_model = fitted_models[best_name]
print("Selected model:", best_name)


Selected model: logistic_regression


In [25]:
# MLflow experiment tracking: log each model's parameters, CV/holdout metrics,
# and the selected pipeline artifact.
import mlflow
import mlflow.sklearn

mlflow.set_experiment("heart-disease-notebook")
mlflow.set_tracking_uri(f"file://{PROJECT_ROOT / 'mlruns'}")
for _, row in results_df.iterrows():
    name = row["model"]
    estimator = candidates[name]
    with mlflow.start_run(run_name=name):
        mlflow.log_param("model", name)
        mlflow.log_param("random_state", RANDOM_STATE)
        mlflow.log_param("cv_folds", 5)
        for metric in row.index:
            if metric != "model":
                mlflow.log_metric(metric, float(row[metric]))
        # Added both 'numpy.dtype' and 'sklearn.tree._tree.Tree' to allow saving both types of models safely
        mlflow.sklearn.log_model(
            fitted_models[name],
            artifact_path="pipeline"
        
        )

model_path = PROJECT_ROOT / "models" / "model.joblib"
model_path.parent.mkdir(parents=True, exist_ok=True)
import joblib
joblib.dump(best_model, model_path)

metrics_path = PROJECT_ROOT / "reports" / "notebook_model_metrics.json"
metrics_path.parent.mkdir(parents=True, exist_ok=True)
metrics_path.write_text(json.dumps({
    "selected_model": best_name,
    "selection_metric": "mean cross-validation ROC-AUC",
    "results": results_df.to_dict(orient="records")
}, indent=2))
print("Saved model:", model_path)
print("Saved metrics:", metrics_path)

2026/10/18 16:53:26 INFO mlflow.tracking.fluent: Experiment with name 'heart-disease-notebook' does not exist. Creating a new experiment.
2026/10/18 16:53:32 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/10/18 16:53:38 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Saved model: /home/s2025ae05678/workspace/2025ae05678-mlops-assignment1/models/model.joblib
Saved metrics: /home/s2025ae05678/workspace/2025ae05678-mlops-assignment1/reports/notebook_model_metrics.json


In [13]:
from pathlib import Path
from google.colab import files

model_path = Path(
    "/content/2025ae05678-mlops-assignment1/models/model.joblib"
)

print("Model exists:", model_path.exists())

if model_path.exists():
    files.download(str(model_path))
else:
    print("Model not found. Current directory:", Path.cwd())

Model exists: True


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Further work to document

- Tune model hyperparameters and record the search space.
- Compare the selected metric against the actual assignment/business objective.
- Add confidence intervals or repeated cross-validation if appropriate.
- Inspect false positives and false negatives, and document limitations.
- Open MLflow UI from the project root with `mlflow ui --backend-store-uri ./mlruns`.
